In [1]:
import os 
import polars as pl
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import roc_auc_score, log_loss, classification_report
import matplotlib.pyplot as plt

os.makedirs("model", exist_ok=True)
DATA_PATH = "train_ranking_data.parquet"

In [2]:
train = pl.scan_parquet(DATA_PATH)
train.collect_schema()

Schema([('customer_id', Int32),
        ('item_id', String),
        ('label', Int8),
        ('i_category_l1', String),
        ('i_category_l2', String),
        ('i_brand', String),
        ('i_price', Float64),
        ('i_item_type', String),
        ('i_age_group', String),
        ('i_gender_target', String),
        ('i_past_orders_count', UInt32),
        ('i_past_units_sold', Int32),
        ('i_past_unique_buyers', UInt32),
        ('u_gender', String),
        ('u_province', String),
        ('u_region', String),
        ('u_membership', String),
        ('u_account_age_days', Int64),
        ('u_past_orders_count', UInt32),
        ('u_past_total_quantity', Int32),
        ('u_past_total_spend', Decimal(precision=38, scale=4)),
        ('u_discount_ratio', Float64),
        ('u_days_since_last_order', Int64),
        ('u_unique_items_bought', UInt32),
        ('u_avg_item_price', Float64),
        ('u_avg_order_value', Float64),
        ('ui_past_purchase_count', UInt32),


In [3]:
train_df = pl.read_parquet(
    DATA_PATH,
    columns=[
        # Keys & Label
        "customer_id", "item_id", "label",
        # Categorical features
        "i_category_l1", "i_category_l2", "i_brand", "i_item_type", "i_age_group", "i_gender_target",
        "u_gender", "u_province", "u_region", "u_membership",
        # Numerical features
        "i_price", "i_past_orders_count", "i_past_units_sold", "i_past_unique_buyers",
        "u_account_age_days", "u_past_orders_count", "u_past_total_quantity", "u_past_total_spend",
        "u_discount_ratio", "u_days_since_last_order", "u_unique_items_bought", "u_avg_item_price", "u_avg_order_value",
        "ui_past_purchase_count", "ui_past_quantity", "ui_days_since_last_buy",
        "u_cat_l1_buy_count", "u_cat_l2_buy_count", "u_brand_buy_count",
        "ui_price_ratio_user_avg", "ui_price_diff_user_avg", "ui_cat_l1_affinity_ratio", "ui_brand_affinity_ratio",
        "ui_bought_companion", "ui_companion_cooccur_count"
    ]
).with_columns(
    pl.col("u_past_total_spend").cast(pl.Float64)
)

print(train_df["label"].value_counts())

shape: (2, 2)
┌───────┬──────────┐
│ label ┆ count    │
│ ---   ┆ ---      │
│ i8    ┆ u32      │
╞═══════╪══════════╡
│ 1     ┆ 5045914  │
│ 0     ┆ 10091828 │
└───────┴──────────┘


In [4]:
cat_cols = [
    "i_category_l1", "i_category_l2", "i_brand", "i_item_type", "i_age_group", "i_gender_target",
    "u_gender", "u_province", "u_region", "u_membership"
]

num_cols = [
    "i_price", "i_past_orders_count", "i_past_units_sold", "i_past_unique_buyers",
    "u_account_age_days", "u_past_orders_count", "u_past_total_quantity", "u_past_total_spend",
    "u_discount_ratio", "u_days_since_last_order", "u_unique_items_bought", "u_avg_item_price", "u_avg_order_value",
    "ui_past_purchase_count", "ui_past_quantity", "ui_days_since_last_buy",
    "u_cat_l1_buy_count", "u_cat_l2_buy_count", "u_brand_buy_count",
    "ui_price_ratio_user_avg", "ui_price_diff_user_avg", "ui_cat_l1_affinity_ratio", "ui_brand_affinity_ratio", 
    "ui_bought_companion", "ui_companion_cooccur_count"
]

features = cat_cols + num_cols
print(f"Tổng số features đưa vào mô hình: {len(features)}")

pdf = train_df.select(features + ["customer_id", "item_id", "label"]).to_pandas()
for col in cat_cols:
    pdf[col] = pdf[col].astype("category")

unique_users = pdf["customer_id"].unique()
np.random.seed(42)
train_users = set(np.random.choice(unique_users, size=int(len(unique_users) * 0.8), replace=False))

train_mask = pdf["customer_id"].isin(train_users)
val_mask = ~train_mask

X_train, y_train = pdf.loc[train_mask, features], pdf.loc[train_mask, "label"]
X_val, y_val = pdf.loc[val_mask, features], pdf.loc[val_mask, "label"]

print(f"— Tập Train : {len(X_train):,} dòng ({len(train_users):,} users)")
print(f"— Tập Val   : {len(X_val):,} dòng ({len(unique_users) - len(train_users):,} users)")

Tổng số features đưa vào mô hình: 35


— Tập Train : 12,112,275 dòng (749,062 users)
— Tập Val   : 3,025,467 dòng (187,266 users)


In [5]:
model = xgb.XGBClassifier(
    n_estimators=500,            
    learning_rate=0.08,      
    max_depth=7,             
    subsample=0.8,               
    colsample_bytree=0.8,          
    tree_method="hist",           
    device="cuda",               
    enable_categorical=True,       
    eval_metric=["auc", "logloss"],
    early_stopping_rounds=30,      
    random_state=42
)


model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=50
)

[0]	validation_0-auc:0.96291	validation_0-logloss:0.58333	validation_1-auc:0.96269	validation_1-logloss:0.58340
[50]	validation_0-auc:0.97135	validation_0-logloss:0.20700	validation_1-auc:0.97107	validation_1-logloss:0.20796
[100]	validation_0-auc:0.97332	validation_0-logloss:0.19727	validation_1-auc:0.97290	validation_1-logloss:0.19881
[150]	validation_0-auc:0.97429	validation_0-logloss:0.19359	validation_1-auc:0.97372	validation_1-logloss:0.19565
[200]	validation_0-auc:0.97495	validation_0-logloss:0.19109	validation_1-auc:0.97426	validation_1-logloss:0.19359
[250]	validation_0-auc:0.97550	validation_0-logloss:0.18900	validation_1-auc:0.97471	validation_1-logloss:0.19187
[300]	validation_0-auc:0.97583	validation_0-logloss:0.18770	validation_1-auc:0.97495	validation_1-logloss:0.19091
[350]	validation_0-auc:0.97613	validation_0-logloss:0.18656	validation_1-auc:0.97517	validation_1-logloss:0.19006
[400]	validation_0-auc:0.97632	validation_0-logloss:0.18582	validation_1-auc:0.97529	valida

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",'cuda'
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",30
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes fr

In [6]:
print("Đang tính toán dự đoán xác suất trên tập Validation...")
val_scores = model.predict_proba(X_val)[:, 1]

# Tạo DataFrame chứa kết quả dự đoán của từng user
val_eval_df = pd.DataFrame({
    "customer_id": pdf.loc[val_mask, "customer_id"].values,
    "label": y_val.values,
    "score": val_scores
})

# Sắp xếp các sản phẩm của từng khách hàng theo Score giảm dần
val_eval_df = val_eval_df.sort_values(by=["customer_id", "score"], ascending=[True, False])

# Hàm tính toán Ranking Metrics chuyên dụng
def evaluate_ranking_metrics(df, k=10, max_users_eval=50000):
    precisions, ndcgs, hit_rates = [], [], []
    discounts = 1.0 / np.log2(np.arange(2, k + 2))
    
    unique_users = df["customer_id"].unique()
    if len(unique_users) > max_users_eval:
        eval_users = np.random.choice(unique_users, size=max_users_eval, replace=False)
        df = df[df["customer_id"].isin(eval_users)]
        
    for uid, group in df.groupby("customer_id"):
        y_true = group["label"].values
        top_k_true = y_true[:k]
        actual_k = len(top_k_true)
        
        # 1. Precision@K
        hits = np.sum(top_k_true)
        precisions.append(hits / k)
        
        # 2. HitRate@K (Có ít nhất 1 món trúng đích)
        hit_rates.append(1.0 if hits > 0 else 0.0)
        
        # 3. NDCG@K
        dcg = np.sum(top_k_true * discounts[:actual_k])
        ideal_y = np.sort(y_true)[::-1][:k]
        idcg = np.sum(ideal_y * discounts[:len(ideal_y)])
        ndcg = (dcg / idcg) if idcg > 0 else 0.0
        ndcgs.append(ndcg)
        
    return {
        f"Precision@{k}": np.mean(precisions),
        f"HitRate@{k}": np.mean(hit_rates),
        f"NDCG@{k}": np.mean(ndcgs)
    }

print("Đang tính toán các chỉ số Top-10 Ranking...")
metrics = evaluate_ranking_metrics(val_eval_df, k=10)
overall_auc = roc_auc_score(y_val, val_scores)
overall_loss = log_loss(y_val, val_scores)

print("\n" + "="*55)
print("🎯 KẾT QUẢ ĐÁNH GIÁ CHẤT LƯỢNG HỆ THỐNG GỢI Ý:")
print("="*55)
print(f" — ROC-AUC Score : {overall_auc:.4f} (Độ phân biệt tổng thể)")
print(f" — Log Loss     : {overall_loss:.4f}")
print(f" — Precision@10  : {metrics['Precision@10']:.4f} ({metrics['Precision@10']*100:.2f}% gợi ý trúng đích)")
print(f" — HitRate@10    : {metrics['HitRate@10']:.4f} ({metrics['HitRate@10']*100:.2f}% khách tìm thấy món muốn mua trong Top 10)")
print(f" — NDCG@10       : {metrics['NDCG@10']:.4f} (Chất lượng thứ tự sắp xếp)")
print("="*55)

Đang tính toán dự đoán xác suất trên tập Validation...


c:\Users\USER\miniconda3\Lib\site-packages\xgboost\core.py:751: UserWarning: [13:08:46] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Đang tính toán các chỉ số Top-10 Ranking...

🎯 KẾT QUẢ ĐÁNH GIÁ CHẤT LƯỢNG HỆ THỐNG GỢI Ý:
 — ROC-AUC Score : 0.9756 (Độ phân biệt tổng thể)
 — Log Loss     : 0.1885
 — Precision@10  : 0.4064 (40.64% gợi ý trúng đích)
 — HitRate@10    : 1.0000 (100.00% khách tìm thấy món muốn mua trong Top 10)
 — NDCG@10       : 0.9761 (Chất lượng thứ tự sắp xếp)


In [7]:
# Lưu mô hình vào thư mục model/
model_path = "model/xgboost_ranking.json"
model.save_model(model_path)

file_size_mb = os.path.getsize(model_path) / (1024 * 1024)
print(f"💾 Đã lưu mô hình thành công tại: {model_path}")
print(f"📦 Dung lượng file mô hình: {file_size_mb:.2f} MB")

💾 Đã lưu mô hình thành công tại: model/xgboost_ranking.json
📦 Dung lượng file mô hình: 29.30 MB
